# Paper 01 · Rosenblatt's Perceptron

**Citation:** Frank Rosenblatt, “The Perceptron: A Probabilistic Model for Information Storage and Organization in the Brain” (1958).

**Paper:** https://doi.org/10.1037/h0042519

> **Scale gap:** We reproduce the learning rule on tiny synthetic 2-D problems, not the original hardware/biological framing.

## Mathematical Framework

Before reproducing the paper experimentally, work through the relevant mathematical companions:

- [Math 01 · Linear Algebra & Geometry](../../math/01_linear_algebra_geometry.ipynb)
- [Math 07 · Linear Models & Decision Boundaries](../../math/07_linear_models_decision_boundaries.ipynb)

For the paper defense, be able to explain the **objective, derivation, assumptions, and why the reported mechanism should follow mathematically**, not just what the code did.

## Before you read
1. What class of decision boundary can one perceptron represent?
2. What would count as a convincing failure case?
3. Why might convergence depend on the geometry of the data?

## Central claim
A simple adaptive linear unit can learn a separating boundary from examples when the classes are linearly separable.

## Student implementation
The cell is runnable now. After the first run, rewrite the marked update from memory.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path(os.environ.get('COURSE_ROOT', next((str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / 'coursekit').is_dir()), '.'))).resolve()
if not (ROOT / 'coursekit').is_dir():
    raise RuntimeError('Open this notebook in the cloned ai-ml-learning repository.')
sys.path.insert(0, str(ROOT))
from coursekit.runtime import configure
SEED = 0
configure(SEED)


In [ ]:
from coursekit.experiments import Experiment
experiment = Experiment('paper-01_perceptron', {'bootstrap_seed': SEED, 'scope': 'educational mechanism demonstration', 'note': 'Original notebook may use additional explicit seeds; source hash records the exact experiment.'}, source='papers/notebooks/01_perceptron.ipynb')
experiment.capture_figures()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def perceptron_train(X, y, lr=1.0, epochs=50):
    w=np.zeros(X.shape[1]); b=0.0; mistakes=[]
    for _ in range(epochs):
        m=0
        for xi,yi in zip(X,y):
            pred=1 if xi@w+b >= 0 else -1
            if pred != yi:
                # TODO: rewrite these two update lines from the learning rule.
                w += lr*yi*xi
                b += lr*yi
                m += 1
        mistakes.append(m)
        if m==0: break
    return w,b,mistakes

## Reproduce convergence on separable data

In [ ]:
rng=np.random.default_rng(0)
A=rng.normal([-2,-2],.6,(80,2)); B=rng.normal([2,2],.6,(80,2))
X=np.vstack([A,B]); y=np.r_[-np.ones(len(A)),np.ones(len(B))]
w,b,m=perceptron_train(X,y)
print("epochs:",len(m),"final mistakes:",m[-1])
plt.plot(m,marker="o"); plt.xlabel("epoch"); plt.ylabel("mistakes"); plt.title("Perceptron convergence"); plt.show()

## Figure-inspired boundary plot

In [ ]:
xx=np.linspace(X[:,0].min()-1,X[:,0].max()+1,200)
yy=-(w[0]*xx+b)/(w[1]+1e-12)
plt.scatter(X[:,0],X[:,1],c=y,cmap="coolwarm",alpha=.6)
plt.plot(xx,yy,"k--"); plt.title("Learned linear boundary"); plt.show()

## Failure case: XOR

In [ ]:
Xxor=np.array([[-1,-1],[-1,1],[1,-1],[1,1]],float)
yxor=np.array([-1,1,1,-1])
_,_,mx=perceptron_train(Xxor,yxor,epochs=30)
plt.plot(mx,marker="o"); plt.title("XOR does not converge"); plt.ylabel("mistakes"); plt.show()
print("final mistakes:",mx[-1])

### Your ablation
Set `b=0` permanently inside training and construct a separable dataset whose boundary does not pass through the origin.

## Ablation table
Fill this after running the experiments.

| Variant | Metric / observation | What changed? | Why? |
|---|---:|---|---|
| Baseline |  |  |  |
| Ablation 1 |  |  |  |
| Ablation 2 |  |  |  |

## Defend the paper
Answer without looking back at the notebook:
1. What problem existed before this work?
2. What was actually new?
3. What evidence in your reproduction supports the central claim?
4. What does your reduced-scale reproduction **not** establish?
5. Which idea from this paper survived into modern systems?
6. What experiment would you run next?

## Evidence export

Figures and numeric diagnostics are captured. Explicit metrics use `experiment.log(variant, seed, metrics)`. Use `run_trials` for paired-seed ablations. An empty metrics table or `not_run` ablation is incomplete evidence, not success. Interpretations remain your work.

In [ ]:
print('Evidence directory:', experiment.finish(globals()))